# IOAI — 2026 Regional 9 10 Asteroids (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-asteroids.zip', '/tmp/ast.zip')
    with zipfile.ZipFile('/tmp/ast.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train_data.csv' if os.path.exists('data/train_data.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Asteroids — 위험 소행성 분류 (RoAI 2026 Regional 9-10)

소행성 표 데이터(헤더 깨진 행 정리 필요)로:
1. (결정적) train 의 `Class` 라벨 반환
2. (ML) test `Class` 이진 분류 — F1(pos=1) 로 채점

**제출**: `submission.csv` 롱포맷 `datapointID, subtaskID, answer`.
**채점(로컬)**: ST2 train held-out F1. **데이터**: `data/train_data.csv`(+`data/test_data.csv`).
원본: judge.nitro-ai.org/competitions (ROAI etapa-judeteana IX-X)

In [ ]:
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

root_path = "data"
train = pd.read_csv(f"{root_path}/train_data.csv")
test = pd.read_csv(f"{root_path}/test_data.csv")
train.head()

In [ ]:
# ST1 (결정적): train 의 Class 라벨 반환
subtask1 = train["Class"]
# ST2 (ML, 채점 대상): X,Y,Z 로 Class 이진분류 — 간단 베이스라인(표준화 KNN).
# 선형모델은 공간 구조를 못 잡아 0점이 되기 쉽습니다. 모범답안은 SVC(RBF)로 비선형 경계를 잡아 F1 을 올립니다.
feat = ["X","Y","Z"]
clf = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(train[feat], train["Class"])
subtask2 = clf.predict(test[feat])

In [ ]:
# 제출(롱포맷): datapointID, subtaskID, answer
def build_sub(sid, ans):
    src = test if sid == 2 else train
    return pd.DataFrame({"datapointID": src["datapointID"], "subtaskID": sid, "answer": np.asarray(ans)})
submission = pd.concat([build_sub(1, subtask1), build_sub(2, subtask2)])
submission.to_csv(f"{root_path}/submission.csv", index=False)
submission.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)